# 01 — EDA و Audit دیتاست استانداردشده

این نوت‌بوک **دیتاست-مستقل** است: روی هر فایل خروجی `standardize.py` (اسکیمای کنونیکال ۱۵ ستونی) کار می‌کند.

کافی است در سلول بعد، نام دیتاست را در `DATASET` بنویسی: `dataset1` / `dataset2`

مراحل (طبق پلن بازتولید مقاله BMC Bioinformatics 2023 — Abnoosian et al.):
1. بارگذاری و چک اسکیما (ترتیب ستون‌ها، کد Gender، برچسب سه‌کلاسه)
2. Audit: duplicate (سطح کامل و سطح فیچر)، گمشده‌ها، outlier های فیزیولوژیک
3. آمار توصیفی + مقایسه با Table 1 مقاله (برای dataset2 = IPDD)
4. توزیع کلاس‌ها و عدم‌توازن
5. همبستگی فیچرها و رابطه‌شان با کلاس
6. خروجی: خلاصه‌ی audit به `outputs/<DATASET>_audit.json`

In [ ]:
# Dataset selector — change this line only
DATASET = "dataset2"   # "dataset1" | "dataset2"

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data") / f"{DATASET}_standard.csv"
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)

CANONICAL = ["ID","Gender","AGE","Urea","Cr","FBS","HbA1c","Chol","TG","HDL","LDL","VLDL","BMI","CLASS_code","CLASS"]
FEATURES = ["Gender","AGE","Urea","Cr","FBS","HbA1c","Chol","TG","HDL","LDL","VLDL","BMI"]

df = pd.read_csv(DATA, encoding="utf-8-sig")
assert list(df.columns) == CANONICAL, "اسکیما با قالب کنونیکال نمی‌خواند!"
print(f"✓ {DATASET} بارگذاری شد: {df.shape[0]} ردیف × {df.shape[1]} ستون")
df.head()

In [ ]:
# 1) چک اسکیما و کدگذاری‌ها
assert set(df["Gender"].dropna().unique()) <= {0, 1}, "Gender باید 0=زن/1=مرد باشد"
assert set(df["CLASS"].unique()) <= {"N","P","Y"}, "CLASS باید N/P/Y باشد"
print("کدگذاری Gender:", df["Gender"].value_counts().rename(index={0:"زن(0)",1:"مرد(1)"}).to_dict())
print("کدگذاری CLASS:", df["CLASS"].value_counts().reindex(["N","P","Y"]).to_dict())
print("\nستون‌های کاملاً خالی (فیچر موجود نبوده):", [c for c in FEATURES if df[c].isna().all()])

In [ ]:
# 2) Audit: duplicates — two levels (ID-exact vs feature-level)
dup_full = df.duplicated(subset=[c for c in CANONICAL if c not in ("ID",)]).sum()
dup_feat = df.duplicated(subset=FEATURES).sum()
print(f"تکراری کامل (همه ستون‌ها بجز ID): {dup_full}")
print(f"تکراری در سطح ۱۲ فیچر (بدون ID): {dup_feat}")
print("\n→ مقاله ادعا کرده ۷ duplicate حذف کرده؛ این عدد را برای dataset2 با ۱۷۴ گزارش قبلی ما مقایسه کن.")

In [ ]:
# 3) گمشده‌ها
miss = df[FEATURES].isna().sum()
miss_pct = (miss / len(df) * 100).round(2)
miss_tbl = pd.DataFrame({"n_missing": miss, "pct": miss_pct})
print(miss_tbl[miss_tbl.n_missing > 0] if miss_tbl.n_missing.sum() else "هیچ مقدار گمشده‌ای نیست")

# outlier های فیزیولوژیک (طبق دامنه‌های طبیعی آزمایشگاه)
RANGES = {"AGE":(18,100),"Urea":(1,10),"Cr":(30,120),"FBS":(3,8),"HbA1c":(3.5,7),
          "Chol":(2.3,6.2),"TG":(0.4,2.3),"HDL":(0.6,2.6),"LDL":(1.2,4.9),"VLDL":(0.2,1.3),"BMI":(15,40)}
rows = []
for c,(lo,hi) in RANGES.items():
    s = df[c].dropna()
    if len(s)==0: continue
    n_lo, n_hi = (s<lo).sum(), (s>hi).sum()
    if n_lo or n_hi:
        rows.append({"feature":c,"below_normal":int(n_lo),"above_normal":int(n_hi),
                     "min":float(s.min()),"max":float(s.max())})
print("\nخارج از دامنه‌ی طبیعی (صرفاً اطلاعی — طبق مقاله outlier حذف نشده):")
print(pd.DataFrame(rows).to_string(index=False) if rows else "  —")

In [ ]:
# 4) آمار توصیفی (بازتولید Table 1 مقاله) + توزیع کلاس‌ها
desc = df[FEATURES].agg(["mean","std","min","max"]).T.round(2)
# مقادیر Table 1 مقاله فقط برای dataset2 (IPDD) قابل مقایسه‌ی مستقیم است
PAPER_T1 = {"AGE":53.74,"Urea":5.18,"Cr":69.28,"FBS":None,"HbA1c":8.26,"Chol":4.85,
            "TG":2.30,"HDL":1.20,"LDL":2.61,"VLDL":1.84,"BMI":29.43}
if DATASET == "dataset2":
    desc["paper_T1"] = [PAPER_T1.get(c) for c in desc.index]
    desc["Δ"] = (desc["mean"] - desc["paper_T1"]).round(2)
print(desc.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11,4))
df["CLASS"].value_counts().reindex(["N","P","Y"]).plot.bar(ax=ax[0], color=["#4c9f70","#e8b23a","#c0504d"])
ax[0].set_title("Class distribution (N/P/Y)"); ax[0].bar_label(ax[0].containers[0])
df["AGE"].plot.hist(bins=25, ax=ax[1], color="#4472c4")
ax[1].set_title("Age distribution")
plt.tight_layout(); plt.savefig(FIG / f"{DATASET}_overview.png", dpi=110); plt.show()

In [ ]:
# 5) همبستگی فیچرها + رابطه با کلاس
num = [c for c in FEATURES if df[c].notna().sum() > 0]
corr = df[num].corr().round(2)
fig, ax = plt.subplots(figsize=(8.5,7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=90)
ax.set_yticks(range(len(num))); ax.set_yticklabels(num)
for i in range(len(num)):
    for j in range(len(num)):
        ax.text(j, i, corr.iloc[i,j], ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i,j])>0.5 else "black")
plt.colorbar(im, shrink=0.8); ax.set_title(f"Feature correlation — {DATASET}")
plt.tight_layout(); plt.savefig(FIG / f"{DATASET}_corr.png", dpi=110); plt.show()

print("میانگین فیچرها به تفکیک کلاس (نشان می‌دهد کدام فیچر قدرت تفکیک دارد):")
print(df.groupby("CLASS")[num].mean().T.round(2).to_string())

In [ ]:
# 6) ذخیره‌ی خلاصه‌ی audit
audit = {
  "dataset": DATASET,
  "n_rows": int(len(df)),
  "n_features_available": int(sum(df[c].notna().any() for c in FEATURES)),
  "features_missing_entirely": [c for c in FEATURES if df[c].isna().all()],
  "class_distribution": df["CLASS"].value_counts().reindex(["N","P","Y"]).fillna(0).astype(int).to_dict(),
  "gender_0F_1M": df["Gender"].value_counts().sort_index().astype(int).to_dict(),
  "duplicates_feature_level": int(dup_feat),
  "missing_pct": {k: float(v) for k, v in miss_pct.items() if v > 0},
  "mean_features": {k: (None if pd.isna(v) else round(float(v),2)) for k,v in df[FEATURES].mean().items()},
}
(OUT / f"{DATASET}_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(audit, ensure_ascii=False, indent=2))